# Phase 0 — Trade-flow EDA notebook

This notebook is the transparent workings view of Chapter 3's descriptive EDA. The narrative chapter is in `outputs/PHASE0_EDA_REPORT.md`; this notebook shows step by step how each analytical artefact was produced.

All analyses delegate to `dissertation/src/eda.py` — the same engine used by `run_phase0.py`. Calling each function regenerates its `.csv` / `.tex` / `.png` / `.pdf` artefacts under `outputs/`; the outputs are deterministic and overwrite cleanly on each run.

**Outputs in this notebook regenerate on run.** Committed cell outputs are typically cleared because the saved figures and tables under `outputs/` are the canonical artefacts.


## 1. Setup and imports


In [ ]:
from __future__ import annotations
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# Make dissertation/src importable when the notebook is launched from notebooks/
NB_DIR = Path.cwd().resolve()
DISS_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(DISS_ROOT))

from src import config as cfg
from src import eda

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("ASK source dir :", eda.ASK_DIR)
print("Figures dir    :", cfg.FIGURES)
print("Tables dir     :", cfg.TABLES)
print("Metrics dir    :", cfg.METRICS)
print("Panel window   :", cfg.YEARS[0], "to", cfg.YEARS[-1])
print("Top-N partners :", cfg.TOP_N_PARTNERS)


## 2. Load raw ASK files

Six workbooks from the 2026-04 ASK release: four yearly aggregations (partner, HS sections, HS chapters, BEC categories) and two monthly partner files (one per flow direction).


In [ ]:
files = {
    "yearly partner"  : eda.PARTNER_FILE,
    "yearly sections" : eda.SECTIONS_FILE,
    "yearly chapter"  : eda.CHAPTER_FILE,
    "yearly BEC"      : eda.BEC_FILE,
    "monthly import"  : eda.MONTHLY_IMP,
    "monthly export"  : eda.MONTHLY_EXP,
}
rows = []
for label, path in files.items():
    rows.append({
        "label": label,
        "filename": Path(path).name,
        "exists": Path(path).exists(),
        "size_bytes": Path(path).stat().st_size if Path(path).exists() else None,
    })
pd.DataFrame(rows)


In [ ]:
# Glance at each workbook's raw layout: first sheet, shape, top rows, dtypes.
for label, path in files.items():
    xl = pd.ExcelFile(path)
    sheet = xl.sheet_names[0]
    raw = pd.read_excel(path, sheet_name=sheet, header=None, nrows=6)
    print(f"\n=== {label}  ({Path(path).name}) ===")
    print(f"  sheet      : {sheet}")
    print(f"  full shape : {pd.read_excel(path, sheet_name=sheet, header=None).shape}")
    display(raw.iloc[:5, :8])


## 3. Parse to long format

Every workbook is parsed into a long-format dataframe with an explicit `flow ∈ {import, export}` column. The yearly partner / chapter workbooks use the alternating Import/Export layout (year-forward-fill on row 2, flow labels on row 3); the sections / BEC workbooks use vertical year blocks with two flow columns side by side; the monthly workbooks use a year-month label on row 2 in `YYYYMmm` format.


In [ ]:
partner = eda.parse_partner_full()
sections = eda.parse_sections_full()
chapter = eda.parse_chapter_full()
bec = eda.parse_bec_full()

for label, df, key_cols in [
    ("partner",  partner,  ["iso2", "partner_name"]),
    ("sections", sections, ["hs_section"]),
    ("chapter",  chapter,  ["hs_chapter", "chapter_name"]),
    ("BEC",      bec,      ["bec_category"]),
]:
    n_unique = df[key_cols[0]].nunique()
    print(f"{label:9s}: shape={df.shape}, unique {key_cols[0]}={n_unique}, "
          f"years {df['year'].min()}–{df['year'].max()}")

display(partner.head(4))
display(sections.head(4))


In [ ]:
monthly_imp, anom_imp = eda.parse_monthly_partner(flow="import")
monthly_exp, anom_exp = eda.parse_monthly_partner(flow="export")

print(f"monthly imports : shape={monthly_imp.shape}, anomalies={len(anom_imp)}")
print(f"monthly exports : shape={monthly_exp.shape}, anomalies={len(anom_exp)}")

display(monthly_imp.head(4))

# Parser anomalies, if any (e.g. the 2203M06 typo in the monthly export file)
parser_anomalies = anom_imp + anom_exp
if parser_anomalies:
    print("\nParser anomalies:")
    display(pd.DataFrame(parser_anomalies))
else:
    print("\nNo parser anomalies.")


## 4. Data-quality checks

Null counts, dtype overview, and the parser anomaly list (already extracted above) are surfaced explicitly for transparency. The parsers coerce missing cells to `0.0` rather than `NaN`, so the null counts here check for any silent gaps in the long-format dataframes themselves (not in the source workbooks).


In [ ]:
quality_rows = []
for label, df in [("partner", partner), ("sections", sections),
                  ("chapter", chapter), ("BEC", bec),
                  ("monthly imports", monthly_imp), ("monthly exports", monthly_exp)]:
    n_null = int(df.isna().sum().sum())
    quality_rows.append({
        "dataset": label, "rows": len(df), "cols": df.shape[1],
        "any_nulls": n_null, "n_unique_years": df["year"].nunique(),
    })
pd.DataFrame(quality_rows)


In [ ]:
# Dtype overview
print("partner dtypes :")
print(partner.dtypes)
print("\nmonthly imports dtypes:")
print(monthly_imp.dtypes)


## 5. Reconciliation checks

The four yearly workbooks describe the same trade flows aggregated different ways. If the data is internally consistent, the sums across (year, flow) should agree.

- **HARD** check: partner ↔ HS sections (must agree within ±0.05%).
- **DIAGNOSTIC** checks: partner ↔ HS chapters, partner ↔ BEC, monthly ↔ yearly. Discrepancies are classified as `pass` / `documented(scope)` / `documented(corruption)`.


In [ ]:
rec_hard = eda.reconcile_partner_vs_sections(partner, sections)
print(f"HARD recon — max |pct_delta| = {rec_hard['pct_delta'].abs().max():.5f}%  (threshold 0.05%)")
display(rec_hard.head(6))


In [ ]:
rec_chapbec = eda.reconcile_partner_vs_chapter_and_bec(partner, chapter, bec)
print("chapter status counts:", rec_chapbec["chapter_status"].value_counts().to_dict())
print("BEC     status counts:", rec_chapbec["bec_status"].value_counts().to_dict())
display(rec_chapbec.sort_values("chapter_pct_delta", key=lambda s: s.abs(), ascending=False).head(6))


In [ ]:
rec_monthly = eda.reconcile_monthly_vs_yearly(monthly_imp, monthly_exp, partner)
print("monthly status counts:", rec_monthly["status"].value_counts().to_dict())
display(rec_monthly.sort_values("pct_delta", key=lambda s: s.abs(), ascending=False).head(6))


In [ ]:
# Bonus side-check: is the chapter-file 2018 anomaly confined to 2018?
anom = eda.eda_chapter_anomaly_check(chapter, sections, partner)
print(f"chapter anomaly confined to 2018 only? {anom['anomaly_confined_to_2018']}")
ratios = pd.Series(anom["chapter_vs_partner_ratio_by_year"]).rename("chapter / partner ratio (imports)").to_frame()
display(ratios)


## 6. Annual trade scale

Imports, exports, deficit, and export/import ratio across the 2010–2024 window. The companion figure plots all three series with the November-2018 to April-2020 tariff window shaded.


In [ ]:
annual = eda.annual_totals_by_flow(partner)
display(annual)
Image(filename=str(cfg.FIGURES / "fig_ch3_imports_exports_totals.png"))


## 7. Zero counts and sparsity

Per-flow per-year zero counts on the full 247-partner population. The export side carries appreciably more mass at zero than the import side — relevant for any later predictive specification.


In [ ]:
zeros = eda.zero_count_by_flow(partner)
display(zeros[zeros["year"].isin([2010, 2018, 2024])])

# Quick 2024 summary
zr_2024 = zeros[zeros["year"] == 2024]
for _, r in zr_2024.iterrows():
    print(f"2024 {r['flow']:6s}: {r['n_zero']}/{r['n_total']} zero ({r['zero_rate_pct']:.2f}%)")


## 8. Partner concentration and HHI

Top-K cumulative shares (K ∈ {5, 10, 20, 50}) and the Herfindahl-Hirschman Index, computed for both flows in 2024 and across the full cumulative 2010–2024 window. The HHI uses raw partner shares scaled by 10,000 — the standard convention in trade economics.


In [ ]:
conc = eda.concentration_curves(partner)
display(conc)
Image(filename=str(cfg.FIGURES / "fig_ch3_concentration_curves.png"))


## 9. Top partners and partner asymmetry

Top-20 partners by imports and top-20 by exports for 2024 (combined into one table via outer-merge). Then a per-partner cumulative imbalance ratio in [-1, +1] showing how one-sided each partner relationship is.


In [ ]:
top20 = eda.top_partners_by_flow(partner, n=20, year=2024)
display(top20)


In [ ]:
asy = eda.eda_partner_asymmetry(partner)
print("Partner-class counts:")
print(asy["classification"].value_counts())
display(asy.head(10))
Image(filename=str(cfg.FIGURES / "fig_ch3_partner_asymmetry.png"))


## 10. Sector composition and sector evolution

Top-5 HS sections per flow at three snapshot years (2010, 2018, 2024), followed by the 21-section × 15-year share-of-flow heatmap (two side-by-side panels — imports / exports).


In [ ]:
sec_comp = eda.sector_composition_by_flow(sections)
display(sec_comp)


In [ ]:
sec_evo = eda.sector_evolution_heatmap(sections)
print(f"sector evolution table: {len(sec_evo)} rows = 21 sections × 15 years × 2 flows")
display(sec_evo.head(6))
Image(filename=str(cfg.FIGURES / "fig_ch3_sector_heatmap.png"))


## 11. Serbia annual trajectory

Annual bilateral trade with Serbia (XS) and the three DiD control partners (AL, MK, ME) for both flows, 2010–2024. The companion figure plots both flows in side-by-side panels.


In [ ]:
serbia = eda.serbia_trajectory_both_flows(partner)
# Pivot for readability
display(serbia[serbia["iso2"] == "XS"].pivot_table(
    index="year", columns="flow", values="value_eur_thousands"
))
Image(filename=str(cfg.FIGURES / "fig_ch3_serbia_imports_exports.png"))


## 12. Serbia monthly event-study and breakpoint precision

The monthly event-study (Serbia vs control mean of AL, MK, ME) covers January 2017 to December 2024 in two panels, one per flow. A complementary breakpoint analysis pinpoints the months in candidate windows with the largest month-on-month deviations relative to the 2017–2021 baseline volatility.


In [ ]:
event = eda.serbia_event_study_monthly(monthly_imp, monthly_exp)
print(f"monthly event-study table: {len(event)} rows")
display(event.head(6))
Image(filename=str(cfg.FIGURES / "fig_ch3_serbia_event_study.png"))


In [ ]:
bp = eda.eda_monthly_breakpoint(monthly_imp)
display(bp)
Image(filename=str(cfg.FIGURES / "fig_ch3_serbia_monthly_breakpoint.png"))


## 13. Sector event response

Per-section year-on-year change in share-of-flow, evaluated in event years (2016 SAA, 2018-11 Serbia tariff onset, 2020-04 tariff removal / COVID, 2022 energy shock). Two complementary views:

- **z-score** — change in event year relative to the section's own non-event-year volatility. Catches unusual single-year movements.
- **percentage-point shift** — substantive absolute share movements (`|Δpp| ≥ 2`), regardless of how unusual they were. Catches structural movements in high-volatility sections that the z-score downweights.


In [ ]:
ser = eda.eda_sector_event_response(sections)
print(f"sector event-response table: {len(ser)} (section, flow, event) triples")
print(f"|z| >= 2 : {(ser['z_score'].abs() >= 2.0).sum()} rows")
display(ser.head(10))


In [ ]:
pp = ser[ser["delta_pp"].abs() >= 2.0].sort_values(
    "delta_pp", key=lambda s: s.abs(), ascending=False
).reset_index(drop=True)
print(f"|Δpp| >= 2 : {len(pp)} rows")
display(pp)
Image(filename=str(cfg.FIGURES / "fig_ch3_sector_event_response.png"))


## 14. Export modellability summary

Computed on the existing 113-partner panel (read-only access to `panel_bilateral.parquet`). The metric JSON consolidates effective sample size, export and import zero rates, the log-variance ratio between flows, and counts of modellable / structural-zero partners.

Also surfaced: the panel-construction proposal — top-113 by imports vs by exports, intersection / union, and the share of cumulative export volume already captured by the import-derived panel.


In [ ]:
mod = eda.export_modellability_assessment(partner)
display(pd.Series(mod).to_frame("value"))


In [ ]:
panel_df, panel_summary = eda.panel_construction_proposal(partner)
display(panel_df)
print(f"Import-derived top-{cfg.TOP_N_PARTNERS} panel covers "
      f"{panel_summary['imports_panel_exports_coverage_pct']:.1f}% of cumulative export volume.")


## 15. Final EDA takeaways

1. **Scale.** Imports and exports both expanded roughly threefold across the 2010–2024 window. The trade deficit widened from €1.86B to €5.43B; the export-to-import ratio stayed within an 11–18% band.
2. **Asymmetry.** Exports are more concentrated (HHI 899 vs 712 in 2024) and sparser (47% zero vs 32% zero) than imports. Seventy-one percent of partners are import-dominant or import-only; only 10% are export-dominant or export-only.
3. **Panel adequacy.** The existing import-derived top-113 panel covers 99.6% of cumulative export volume, so the same panel can carry export-side analyses without modification.
4. **Sector dynamics.** Largest structural moves are on the export side. Section 15 (base metals) lost 12.6pp of export share around 2016 and another 11.0pp around 2022; section 5 (mineral products) gained share in both windows. These shifts are temporally aligned with the SAA entering into force (2016) and the 2022 energy shock; the EDA does not establish causality.
5. **Serbia shock.** Monthly imports from Serbia drop precisely in November 2018 and recover precisely in April 2020 — both shifts coincide exactly with the policy reference dates. The export trajectory does not mirror the collapse.
6. **Data caveats.** The chapter (tab03) workbook has a confined 2018 anomaly — sectoral analyses for 2018 should use HS sections (tab04) instead. The monthly export file carries a `2203M06` label typo for June 2023; the row is dropped from the monthly diagnostic. The unspecified-destination residual `ZZ:` row must be retained for partner-vs-sections reconciliation but excluded from per-partner analyses.
7. **Export modellability.** A 32.7% zero rate on the panel and a 1.38× variance ratio versus imports imply a higher error floor for export-side predictive work. PPML is the appropriate baseline; 86 of 113 partners are modellable with >5 non-zero export years.
